Given:

- X ∈ R^{N×D}
- y ∈ R^{N}

Implement linear regression using gradient descent:
- No loops over samples
- Add L2 regularization
- Return learned weights and loss history

Follow-ups:
- Add feature normalization
- Detect divergence / instability
- Convert to closed-form solution

# Problem Formulation
$X \in [B, D_{in}], y \in [B, D_{out}], W \in [D_{in}, D_{out}]$

$y = XW$

The loss function is l2-norm and regulariziation of weights.

$l = \frac{1}{B}||y' - y||^2_2 + \lambda ||W||^2_2$ 

The gradient is then:

$$
\frac{dl}{dW} = \frac{\partial l}{\partial y'}\frac{\partial y'}{\partial W} + \frac{\partial l}{\partial W} \\
    = 2 \frac{1}{B} (y'-y) X + 2\lambda W \\
$$

## Pytorch Implementation

In [15]:
import torch

class Model(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__() 
        self.linear = torch.nn.Linear(in_channels, out_channels)
    
    def forward(self, x):
        return self.linear(x)

# regularization term
def reg_fn(model, lamb):
    reg_loss = 0.0
    for param in model.parameters():
        reg_loss += torch.norm(param, p=2) ** 2
    return lamb * reg_loss

# total loss function
def total_loss_fn(y_pred, y, model, lamb):
    mse_loss = torch.mean((y_pred - y) ** 2)
    reg_loss = reg_fn(model, lamb)
    return mse_loss + reg_loss

# training loop
B, D_in, D_out = 100, 10, 4

# create random input and output data
X = torch.randn(B, D_in)
y = torch.randn(B, D_out)

# training settings
max_iters = 1000
lr = 1e-3
lamb = 0.01

# train
model = Model(D_in, D_out)
loss_fn = total_loss_fn
optimizer = torch.optim.SGD(model.parameters(), lr = lr)

for step in range(max_iters):
    # zero grad 
    optimizer.zero_grad()

    # forward pass 
    y_pred = model(X)

    # loss 
    loss = loss_fn(y_pred, y, model, lamb)

    # backward pass for gradients
    loss.backward() 

    # update weights using gradient descent
    optimizer.step()

    # log 
    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

Step 0, Loss: 1.3943
Step 100, Loss: 1.3378
Step 200, Loss: 1.2888
Step 300, Loss: 1.2463
Step 400, Loss: 1.2094
Step 500, Loss: 1.1772
Step 600, Loss: 1.1492
Step 700, Loss: 1.1247
Step 800, Loss: 1.1033
Step 900, Loss: 1.0846


## Numpy Implementation

In [16]:
import numpy as np
import random

In [17]:
# data/model dimensions
# x: (b, d_in)
# y: (b, d_out)

B, D_in, D_out = 100, 10, 4

# create random input and output data
X = np.random.randn(B, D_in)
y = np.random.randn(B, D_out) 

# model weights 
W = np.random.randn(D_in, D_out)
#b = np.random.randn(D_out)

# training parameters
lamb = 0.01
max_iters = 1000
lr = 1e-04

In [18]:
# HELPER FUNCTIONs
def loss_fn(y_pred, y, W, lamb):
    return np.mean((y_pred - y) ** 2) + lamb * np.linalg.norm(W, 2) ** 2

def grad_fn(y_pred, y, X, W, lamb):
    grad_w = 2 * X.T @ (y_pred - y) + 2 * lamb * W
    return grad_w

In [19]:
# Linear 
for step in range(max_iters):
    y_pred = X @ W #+ b # (B, D_out)

    # loss 
    loss = loss_fn(y_pred, y, W, lamb) 

    # compute gradients
    grad_W = grad_fn(y_pred, y, X, W, lamb)

    # update weights using gradient descent 
    W = W - lr * grad_W
    #b = b - lr * grad_b

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss:.4f}")

Step 0, Loss: 10.0719
Step 100, Loss: 1.1174
Step 200, Loss: 0.9074
Step 300, Loss: 0.8986
Step 400, Loss: 0.8981
Step 500, Loss: 0.8980
Step 600, Loss: 0.8980
Step 700, Loss: 0.8980
Step 800, Loss: 0.8980
Step 900, Loss: 0.8980
